In [21]:
import json
import re
import sqlite3
import os
from collections import defaultdict
import sqlglot
from sqlglot.expressions import Join
from collections import Counter

In [ ]:
SPIDER_DB_DIR = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/database" 
FILE_PATH = "/mnt/storage_C1/igorzwirtes/poster_ic/predictions/predictions_base_2.json"

In [23]:
def normalize_value(v):

    if v is None:
        return None

    if isinstance(v, float):
        return round(v, 6)

    if isinstance(v, str):
        return v.strip().lower()

    return v

def normalize_result(rows):

    normalized = []

    for row in rows:
        normalized.append(
            tuple(normalize_value(v) for v in row)
        )

    return sorted(normalized, key=str)

def execute_query(db_id, sql):

    db_path = os.path.join(
        SPIDER_DB_DIR,
        db_id,
        f"{db_id}.sqlite"
    )

    conn = None

    try:
        conn = sqlite3.connect(db_path)

        cursor = conn.cursor()

        cursor.execute(sql)

        rows = cursor.fetchall()

        return normalize_result(rows)

    except Exception:
        return None

    finally:
        if conn:
            conn.close()

def syntax_valid(db_id, sql):

    db_path = os.path.join(
        SPIDER_DB_DIR,
        db_id,
        f"{db_id}.sqlite"
    )

    conn = None

    try:
        conn = sqlite3.connect(db_path)

        cur = conn.cursor()

        cur.execute(sql)

        return True

    except Exception:
        return False

    finally:
        if conn:
            conn.close()

def execution_accuracy(gold, pred, db_id):

    gold_results = execute_query(db_id, gold)

    pred_results = execute_query(db_id, pred)

    if gold_results is None or pred_results is None:
        return False

    return gold_results == pred_results

def exact_match(gold, pred):

    try:
        gold_ast = sqlglot.parse_one(gold)
        pred_ast = sqlglot.parse_one(pred)

        return gold_ast == pred_ast

    except Exception:
        return False
    
def classify_error(pred_sql, gold_sql):

    pred_lower = pred_sql.lower()
    gold_lower = gold_sql.lower()

    if not pred_sql or pred_sql.strip() == "":
        return "empty"

    if "join" in gold_lower and "join" not in pred_lower:
        return "missing_join"

    if "group by" in gold_lower and "group by" not in pred_lower:
        return "missing_group_by"

    if "having" in gold_lower and "having" not in pred_lower:
        return "missing_having"

    if "intersect" in gold_lower and "intersect" not in pred_lower:
        return "missing_intersect"

    if "except" in gold_lower and "except" not in pred_lower:
        return "missing_except"

    where_pos = gold_lower.find("where")

    if where_pos != -1:

        after_where = gold_lower[where_pos:]

        if "select" in after_where:

            pred_after_where = pred_lower[
                pred_lower.find("where"):
            ]

            if "select" not in pred_after_where:
                return "missing_subquery"

    if "order by" in gold_lower and "order by" not in pred_lower:
        return "missing_order_by"

    return "wrong_columns_or_values"

In [24]:
with open(FILE_PATH) as f:
    predictions = json.load(f)

In [25]:
evaluated_predictions = []

for p in predictions:

    syntax_ok = syntax_valid(
        p["db_id"],
        p["predicted"]
    )

    ex_acc = execution_accuracy(
        p["gold"],
        p["predicted"],
        p["db_id"]
    )

    em = exact_match(
        p["gold"],
        p["predicted"]
    )

    error_type = None

    if not ex_acc:

        error_type = classify_error(
            p["predicted"],
            p["gold"]
        )

    evaluated_predictions.append({
        **p,
        "syntax_valid": syntax_ok,
        "execution_accuracy": ex_acc,
        "exact_match": em,
        "error_type": error_type
    })

In [26]:
total = len(evaluated_predictions)

syntax_score = sum(
    p["syntax_valid"]
    for p in evaluated_predictions
) / total

execution_score = sum(
    p["execution_accuracy"]
    for p in evaluated_predictions
) / total

exact_match_score = sum(
    p["exact_match"]
    for p in evaluated_predictions
) / total

print("\n" + "=" * 80)
print("GLOBAL METRICS")
print("=" * 80)

print(f"Total Examples:      {total}")
print(f"Syntax Validity:     {syntax_score:.4f}")
print(f"Execution Accuracy:  {execution_score:.4f}")
print(f"Exact Match (AST):   {exact_match_score:.4f}")


GLOBAL METRICS
Total Examples:      1034
Syntax Validity:     0.9487
Execution Accuracy:  0.7427
Exact Match (AST):   0.2108


In [27]:
invalid_queries = [
    p for p in evaluated_predictions
    if not p["syntax_valid"]
]

print("\n" + "=" * 80)
print("INVALID SQL")
print("=" * 80)

print(f"Invalid Queries: {len(invalid_queries)}")

for p in invalid_queries[:5]:

    print("\nQuestion:")
    print(p["question"])

    print("\nPredicted:")
    print(p["predicted"])

    print("-" * 80)


INVALID SQL
Invalid Queries: 53

Question:
Find the last name of the student who has a cat that is age 3.

Predicted:
SELECT T2.LName FROM Has_Pet AS T1 JOIN Student AS T2 ON T1.StuID = T2.StuID JOIN Pets AS T3 ON T1.PetID = T3.PetID WHERE T3.PetAge = 3 AND T3.PetType = 'cat'
--------------------------------------------------------------------------------

Question:
What is the last name of the student who has a cat that is 3 years old?

Predicted:
SELECT T2.LName FROM Pets AS T1 JOIN Student AS T2 ON T1.PetID = T2.StuID WHERE T1.PetAge = 3 AND T1.PetType = 'cat'
--------------------------------------------------------------------------------

Question:
Which model of the car has the minimum horsepower?

Predicted:
SELECT T2.Model FROM cars_data AS T1 INNER JOIN car_names AS T2 ON T1.MakeId = T2.MakeId ORDER BY T1.Horsepower ASC LIMIT 1
--------------------------------------------------------------------------------

Question:
What is the model of the car with the smallest amount of h

In [28]:
errors = [
    p for p in evaluated_predictions
    if not p["execution_accuracy"]
]

error_counter = Counter(
    p["error_type"]
    for p in errors
)

print("\n" + "=" * 80)
print("ERROR DISTRIBUTION")
print("=" * 80)

for error_type, count in error_counter.most_common():

    percentage = 100 * count / len(errors)

    print(
        f"{error_type:<30} "
        f"{count:<5} "
        f"({percentage:.2f}%)"
    )


ERROR DISTRIBUTION
wrong_columns_or_values        198   (74.44%)
missing_join                   33    (12.41%)
missing_intersect              11    (4.14%)
missing_subquery               9     (3.38%)
missing_group_by               8     (3.01%)
missing_order_by               4     (1.50%)
missing_except                 3     (1.13%)


In [29]:
grouped_errors = defaultdict(list)

for p in errors:
    grouped_errors[p["error_type"]].append(p)

In [30]:
for error_type, examples in grouped_errors.items():

    print("\n" + "=" * 80)
    print(f"ERROR TYPE: {error_type}")
    print(f"COUNT: {len(examples)}")
    print("=" * 80)

    for i, p in enumerate(examples[:5], 1):

        print(f"\nExample {i}")

        print("\nQuestion:")
        print(p["question"])

        print("\nGold SQL:")
        print(p["gold"])

        print("\nPredicted SQL:")
        print(p["predicted"])

        print("-" * 80)


ERROR TYPE: wrong_columns_or_values
COUNT: 198

Example 1

Question:
What is the maximum capacity and the average of all stadiums ?

Gold SQL:
select max(capacity), average from stadium

Predicted SQL:
SELECT max(Capacity) ,  avg(Capacity) FROM stadium
--------------------------------------------------------------------------------

Example 2

Question:
What is the average and maximum capacities for all stadiums ?

Gold SQL:
select avg(capacity) ,  max(capacity) from stadium

Predicted SQL:
SELECT max(Capacity) ,  avg(Capacity) FROM stadium
--------------------------------------------------------------------------------

Example 3

Question:
Find the maximum weight for each type of pet. List the maximum weight and pet type.

Gold SQL:
SELECT max(weight) ,  petType FROM pets GROUP BY petType

Predicted SQL:
SELECT PetType, MAX(weight) FROM Pets GROUP BY PetType
--------------------------------------------------------------------------------

Example 4

Question:
List the maximum weight